# Peningkatan SVM — fitur & ensemble  (+ evaluasi cross-validation)

Eksperimen meningkatkan SVM (track **B**) dievaluasi secara **robust dengan 5-fold CV**
(track **D**), pada **v2 (balanced 3k)** — versi terbaik SVM.

Teknik yang diuji:
1. word TF-IDF + LinearSVC (baseline)
2. **word+char TF-IDF** + LinearSVC  *(char n-gram tahan typo/alay)*
3. word + ComplementNB
4. word+char + LogisticRegression
5. **Ensemble** (LinearSVC + ComplementNB + LogReg, hard voting)

> **Catatan penting (track D):** macro-F1 CV (~0,59–0,62) lebih **jujur** daripada
> single-split (0,694 yg optimistik). Laporkan angka CV (mean±std).

## 0. Dependency

In [1]:
%pip install -q "pymongo[srv]" dnspython certifi python-dotenv scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Baca v2 (balanced 3k) dari `processed_svm`

In [2]:
import os, numpy as np, pandas as pd
from pymongo import MongoClient
import certifi
MONGO_URI=os.environ.get("MONGO_URI","")
if not MONGO_URI:
    try:
        from dotenv import load_dotenv; load_dotenv(); MONGO_URI=os.environ.get("MONGO_URI","")
    except Exception: pass
if not MONGO_URI:
    from getpass import getpass; MONGO_URI=getpass("MONGO_URI: ")
DB=os.environ.get("MONGO_DB_NAME","youtube_sentiment")
client=MongoClient(MONGO_URI,tlsCAFile=certifi.where(),serverSelectionTimeoutMS=20000); client.admin.command("ping")
df=pd.DataFrame(list(client[DB]["processed_svm"].find({"in_balanced_set":True},{"_id":0,"svm":1,"label_id":1})))
df=df[df["svm"].astype(str).str.len()>0]
X=df["svm"].astype(str).tolist(); y=np.asarray(df["label_id"],dtype=int)   # list+numpy (pandas3/pyarrow)
print(f"v2: {len(X)} dok")

v2: 2998 dok


## 2. Definisi pipeline & teknik

In [3]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
SEED=42
def wv():    return TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,min_df=2)
def union(): return FeatureUnion([("w",TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,min_df=2)),
                                  ("c",TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),sublinear_tf=True,min_df=3))])
techniques={
 "1 word+LinearSVC (baseline)": Pipeline([("v",wv()),("c",LinearSVC(class_weight="balanced",random_state=SEED))]),
 "2 word+char+LinearSVC":       Pipeline([("v",union()),("c",LinearSVC(class_weight="balanced",random_state=SEED))]),
 "3 word+ComplementNB":         Pipeline([("v",wv()),("c",ComplementNB())]),
 "4 word+char+LogReg":          Pipeline([("v",union()),("c",LogisticRegression(max_iter=2000,class_weight="balanced",C=2.0))]),
 "5 Ensemble (SVC+NB+LogReg)":  Pipeline([("v",union()),("c",VotingClassifier(
       [("svc",LinearSVC(class_weight="balanced",random_state=SEED)),("nb",ComplementNB()),
        ("lr",LogisticRegression(max_iter=2000,class_weight="balanced"))],voting="hard"))]),
}

## 3. Evaluasi 5-fold cross-validation (robust)

In [4]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv=StratifiedKFold(5,shuffle=True,random_state=SEED); res={}
print(f"{'Teknik':30s} macro-F1 (5-fold CV)")
print("-"*56)
base=None
for name,pipe in techniques.items():
    s=cross_val_score(pipe,X,y,cv=cv,scoring="f1_macro",n_jobs=-1); res[name]=(s.mean(),s.std())
    if base is None: base=s.mean()
    print(f"{name:30s} {s.mean():.4f} ± {s.std():.4f}  ({(s.mean()-base)*100:+.1f}%)")

Teknik                         macro-F1 (5-fold CV)
--------------------------------------------------------


1 word+LinearSVC (baseline)    0.5915 ± 0.0223  (+0.0%)


2 word+char+LinearSVC          0.5959 ± 0.0174  (+0.4%)


3 word+ComplementNB            0.5847 ± 0.0099  (-0.7%)


4 word+char+LogReg             0.6129 ± 0.0267  (+2.1%)


5 Ensemble (SVC+NB+LogReg)     0.6180 ± 0.0241  (+2.7%)


## 4. Verifikasi di test split kanonik v2 (nyambung ke baseline 0,694)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
raw=pd.DataFrame(list(client[DB]["raw_comments"].find({"in_balanced_set":True},{"_id":0,"comment_id":1,"label":1})))
LAB=["Negatif","Netral","Positif"]; raw["label_id"]=raw["label"].map({l:i for i,l in enumerate(LAB)})
sv=pd.DataFrame(list(client[DB]["processed_svm"].find({},{"_id":0,"comment_id":1,"svm":1})))
d=raw.merge(sv,on="comment_id",how="left"); d["svm"]=d["svm"].fillna(""); d=d.sort_values("comment_id").reset_index(drop=True)
tmp,te=train_test_split(d,test_size=.10,stratify=d["label_id"],random_state=SEED); tr=tmp[tmp["svm"].str.len()>0]
Xtr,ytr=tr["svm"].astype(str).tolist(),np.asarray(tr["label_id"],int)
Xte,yte=te["svm"].astype(str).tolist(),np.asarray(te["label_id"],int)
for name in ["1 word+LinearSVC (baseline)","5 Ensemble (SVC+NB+LogReg)"]:
    m=techniques[name]; m.fit(Xtr,ytr); p=m.predict(Xte)
    print(f"{name:30s} test macro-F1 {f1_score(yte,p,average='macro'):.4f} | acc {accuracy_score(yte,p):.4f}")

1 word+LinearSVC (baseline)    test macro-F1 0.6668 | acc 0.6667


5 Ensemble (SVC+NB+LogReg)     test macro-F1 0.6732 | acc 0.6733


## 5. Simpan ringkasan

In [6]:
import json,pathlib
ROOT=pathlib.Path.cwd()
for _p in [ROOT,*ROOT.parents]:
    if (_p/"configs").exists() or (_p/".git").exists(): ROOT=_p; break
REP=ROOT/"outputs"/"reports"; REP.mkdir(parents=True,exist_ok=True)
json.dump({f"{k}":{"cv_macro_f1":round(v[0],4),"cv_std":round(v[1],4)} for k,v in res.items()},
          open(REP/"svm_improve_cv.json","w"),ensure_ascii=False,indent=2)
print("Tersimpan: outputs/reports/svm_improve_cv.json")

Tersimpan: outputs/reports/svm_improve_cv.json


## Kesimpulan

- **Ensemble word+char (SVC+NB+LogReg)** = teknik terbaik (~+2–3% vs baseline).
- **CV mengoreksi** angka single-split yang optimistik → laporkan macro-F1 CV (mean±std).
- Gain modest → konsisten dgn hipotesis bahwa **plafon ada di kualitas label** (track A)
  dan **domain model** (track C: IndoBERTweet).